In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data
X_train_df = pd.read_csv('../data/raw/X_train_update.csv', index_col=0)
classe_label = pd.read_csv("../data/raw/product_class.csv", sep=';')
df_classes = pd.read_csv('../data/raw/Y_train_CVw08PX.csv', index_col=0)

print("shape of X_train_df:", X_train_df.shape)
print("shape of df_classes:", df_classes.shape)
print("shape of classe_label:", classe_label.shape)


shape of X_train_df: (84916, 4)
shape of df_classes: (84916, 1)
shape of classe_label: (27, 3)


## Modélisation

Dans le cadre de cette étude, un pipeline complet de préparation des données sera mis en place : **nettoyage et normalisation des textes**, **standardisation des descriptions**, ainsi que **structuration des chemins d’images** pour l’extraction visuelle.  
Ces étapes permettront de constituer un dataset exploitable pour la phase de modélisation.

Le dataset brut contient **84 916 échantillons**, organisés comme suit :

* **4 colonnes d’entrée** prévues pour la modélisation (`X_train_df`, shape : 84 916 × 4)
* **1 colonne cible** représentant la catégorie produit (`df_classes`, shape : 84 916 × 1)
* **27 classes uniques**, décrites dans la table de référence (`classe_label`, shape : 27 × 3)

Cette configuration pose une tâche de classification multiclasses, caractérisée par un **déséquilibre important entre les catégories**, un élément qui devra être explicitement pris en compte lors de la modélisation.  

L’objectif de cette phase sera d’**extraire des représentations pertinentes** (texte + image), puis d’entraîner un modèle supervisé performant, en intégrant des méthodes adaptées au déséquilibre observé dans les données.  


### Data preparation

In [13]:
df_train_classes = X_train_df.merge(df_classes, left_index=True, right_index=True, how='left')
df_train_classes.head()

,designation,description,productid,imageid,prdtypecode
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,1280
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705


In [14]:
df_txt_classes = df_train_classes.merge(classe_label, left_on='prdtypecode', right_on='prdtypecode', how='left')
df_txt_classes.head()

,designation,description,productid,imageid,prdtypecode,target,prodtype
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10,0,Livres adultes
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280,15,Magazines
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,25,Accessoires jeux vidéo
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,1280,4,Jouets enfance
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,22,Livres illustrés


In [15]:
import re
import os

images = []
image_dir = '../data/raw/images/image_train/'
for filename in os.listdir(image_dir):
    img = m = re.match(r"image_(\d+)_product_(\d+)\.jpg", filename)
    if filename.endswith('.jpg') :
        image_id = int(re.findall(r'(\d+)', filename)[0])
        product_id = int(re.findall(r'(\d+)', filename)[1])
        path_image = os.path.join(image_dir, filename)
        images.append({'imageid': image_id, 'productid': product_id, 'filename': filename, 'path_image': path_image})

df_images = pd.DataFrame(images)

In [16]:
# merge txt and image data
df_merge = df_txt_classes.merge(df_images, left_on=['productid','imageid'], right_on=['productid','imageid'], how='left')
print("shape of df_merge:", df_merge.shape)
df_merge.head()

shape of df_merge: (84916, 9)


,designation,description,productid,imageid,prdtypecode,target,prodtype,filename,path_image
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10,0,Livres adultes,image_1263597046_product_3804725264.jpg,../data/raw/images/image_train/image_126359704...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280,15,Magazines,image_1008141237_product_436067568.jpg,../data/raw/images/image_train/image_100814123...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,25,Accessoires jeux vidéo,image_938777978_product_201115110.jpg,../data/raw/images/image_train/image_938777978...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,1280,4,Jouets enfance,image_457047496_product_50418756.jpg,../data/raw/images/image_train/image_457047496...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,22,Livres illustrés,image_1077757786_product_278535884.jpg,../data/raw/images/image_train/image_107775778...


In [17]:
df_model = df_merge.drop(columns=['filename', 'target', 'prdtypecode'] , inplace=False , axis=1)
df_model.head()

,designation,description,productid,imageid,prodtype,path_image
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,Livres adultes,../data/raw/images/image_train/image_126359704...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,Magazines,../data/raw/images/image_train/image_100814123...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,Accessoires jeux vidéo,../data/raw/images/image_train/image_938777978...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,Jouets enfance,../data/raw/images/image_train/image_457047496...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,Livres illustrés,../data/raw/images/image_train/image_107775778...


In [18]:
# suppress URLs and HTML tags & normalize text
import re
import html

def clean_text(text: str) -> str:
    if text is None or pd.isna(text):
        return ""
    text = html.unescape(str(text)).lower()

    # supprimer HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # supprimer URLs + emails 
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", text)

    # supprimer codes produits
    text = re.sub(
        r"\b(ref|réf|reference|référence|fabricant|model|modèle|sku|asin|ean|isbn|gtin)[\s\.:_-]*[a-z0-9][a-z0-9\-_.]{2,}\b",
        " ",
        text
    )
    text = re.sub(r"[a-zA-Z]{2,}-\d+(?:-[a-zA-Z]{2,})?", " ", text)

    # supprimer nombres longs
    text = re.sub(r"\b\d{6,}\b", " ", text)
     
    # nettoyage final
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [19]:
df_model['text'] = df_model['designation'].fillna("").apply(clean_text) + " " + df_model['description'].fillna("").apply(clean_text)
df_model = df_model.sort_values("productid").reset_index(drop=True)
df_model = df_model.drop(columns=['designation', 'description'], axis=1)
print("shape of df_model after text cleaning:", df_model.shape)
df_model.head()

shape of df_model after text cleaning: (84916, 5)


,productid,imageid,prodtype,path_image,text
0,183912,528916,Jeux vidéo import,../data/raw/images/image_train/image_528916_pr...,matchbox caterpillar construction zone gbc
1,184080,1086436,Jeux vidéo import,../data/raw/images/image_train/image_1086436_p...,maya l'abeille (compatible game boy)
2,184251,234234,Jeux vidéo import,../data/raw/images/image_train/image_234234_pr...,toonstein : le château hante
3,184334,669302,Jeux vidéo import,../data/raw/images/image_train/image_669302_pr...,army men toys in space
4,184381,669220,Jeux vidéo import,../data/raw/images/image_train/image_669220_pr...,international track & field


### Extract features
> **Texte** : embeddings MPNet > **all-mpnet-base-v2**  
> **Image** : embeddings EfficientViT-B2 > **efficientvit_b2.r288_in1k**


In [62]:
# extract features with Mpnet for text
from sentence_transformers import SentenceTransformer
import torch

DEVICE_TXT = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE_TXT = 64 if DEVICE_TXT == 'cpu' else 128
MODEL_NAME_TXT = 'sentence-transformers/all-mpnet-base-v2'


def extract_text_features(
    df: pd.DataFrame, 
    text_column: str = 'text', 
    model_name: str = MODEL_NAME_TXT,
    batch_size: int = BATCH_SIZE_TXT,
    device: str = DEVICE_TXT
    ) -> np.ndarray:
    
    """
    Extracts MPNet embeddings for a given text column and returns a DataFrame.

    Parameters:
        df (pd.DataFrame): Input dataframe
        text_column (str): Column name containing cleaned text
        model_name (str): SentenceTransformer model
        batch_size (int): Batch size for CPU extraction
        device (str): "cpu" or "cuda"

    Returns:
        np.ndarray: Array with text embeddings
    """
    # Charger modèle sur CPU
    model = SentenceTransformer(model_name, device=device)
    
    # Récupérer les textes
    texts = df[text_column].tolist()
    
    # Encodage batché 
    embeddings = model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=True, 
        normalize_embeddings=True
    )
    return embeddings


In [63]:
embeddings_text = extract_text_features(df_model, text_column='text')

# save text embeddings in numPy array
np.save('../data/processed/embeddings_text_mpnet.npy', embeddings_text)
print("shape of text embeddings:", embeddings_text.shape)


embeddings_text_df = pd.DataFrame(embeddings_text, index=df_model.index)
# save text embeddings in DataFrame
embeddings_text_df.to_csv('../data/processed/embeddings_text_mpnet.csv')
embeddings_text_df.head()

Batches: 100%|██████████| 1327/1327 [1:12:24<00:00,  3.27s/it]


shape of text embeddings: (84916, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.005569,-0.026126,0.023353,0.034011,-0.033499,-0.005654,-0.005498,0.006786,-0.053304,0.034759,...,-0.023706,-0.019857,-0.021738,-0.017378,0.033908,0.005519,-0.009981,-0.031828,-0.084632,0.015145
1,-0.015285,0.058520,-0.001478,0.050197,0.021181,0.023596,0.021534,0.001994,-0.038479,0.048774,...,-0.041353,0.054392,-0.030574,-0.012782,0.013212,0.071525,0.002619,-0.011347,-0.022455,0.003433
2,0.047774,0.032098,0.017832,0.034363,-0.064152,-0.004246,0.023759,0.020144,0.015139,0.008468,...,-0.066552,0.004509,-0.058296,0.027578,0.000449,0.046898,-0.013900,0.010245,0.022519,-0.012628
3,0.068436,-0.023975,0.015142,-0.022647,0.011989,-0.013772,0.068468,-0.016179,-0.017423,0.010122,...,-0.003371,-0.037485,-0.005276,0.006936,0.028107,0.040790,0.017235,0.014587,0.053040,-0.024506
4,-0.077636,-0.026529,-0.020417,0.072591,0.012251,-0.013279,-0.000506,0.008971,0.020327,-0.026854,...,-0.039680,0.040178,0.067530,-0.006398,-0.000526,-0.021796,-0.006233,-0.010095,0.046712,0.003736


In [ ]:
# extract features image with EfficientViT-B2
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
import timm
from tqdm.auto import tqdm

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE_IMAGE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE_IMAGE = 64 if DEVICE_IMAGE == 'cpu' else 128
MODEL_NAME_IMAGE = "efficientvit_b2.r288_in1k"   # EfficientViT-B2

class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, path_column: str, transform):
        self.df = df.reset_index(drop=True)
        self.path_column = path_column
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx][self.path_column]
        try:
            image = Image.open(path).convert("RGB")
        except Exception as e:
            # image blanche en cas d'erreur de chargement   
            image = Image.new("RGB", (224, 224), (255, 255, 255))
            
        img = self.transform(image)
        return img


def extract_image_features(
    df: pd.DataFrame, 
    path_column: str = 'path_image', 
    batch_size: int = BATCH_SIZE_IMAGE,
    device: str = DEVICE_IMAGE
    ) -> np.ndarray:
    
    """
    Extracts EfficientViT-B2 embeddings for images and returns a numpy array

    Parameters:
        df (pd.DataFrame): Input dataframe
        path_column (str): Column name containing image paths
        batch_size (int): Batch size for CPU extraction
        model_name (str): timm model name
        device (str): "cpu" or "cuda"
    
    Returns:
        np.ndarray: Array with image embeddings
    """
    # Charger modèle
    model = timm.create_model(MODEL_NAME_IMAGE, pretrained=True, num_classes=0)
    model.to(device)
    model.eval()
    
    # Prétraitement des images
    model_cfg = timm.data.resolve_model_data_config(model)
    transform = timm.data.create_transform(**model_cfg)
    
    dataset = ImageDataset(df, path_column=path_column, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # Extraction des features
    features_list = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting image features",total=len(loader)):
            batch = batch.to(device, non_blocking=True)
            feats = model(batch)       
            features_list.append(feats.cpu().numpy())
            
    images_features = np.vstack(features_list)
    return images_features

In [65]:
embeddings_image = extract_image_features(df_model, path_column='path_image')
# save image embeddings in numPy array
np.save('../data/processed/embeddings_image_efficientnetv2s.npy', embeddings_image)

print("shape of image embeddings:", embeddings_image.shape)

embeddings_image_df = pd.DataFrame(embeddings_image, index=df_model.index)
# save image embeddings in DataFrame
embeddings_image_df.to_csv('../data/processed/embeddings_image_efficientnetv2s.csv')
embeddings_image_df.head()

Extracting image features: 100%|██████████| 1327/1327 [47:34<00:00,  2.15s/it]


shape of image embeddings: (84916, 2560)


,0,1,2,3,4,5,6,7,8,9,...,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559
0,-0.349980,-0.000016,1.047091e-04,-0.263927,-0.028223,-0.026561,0.000033,-0.0,-0.0,-0.049372,...,0.053803,-0.000000,-0.000048,-0.074203,0.013956,-0.003005,-0.348766,-0.04569,0.347379,0.000063
1,0.854163,-0.000140,4.417914e-07,-0.281158,0.285039,0.052766,0.000026,-0.0,-0.0,-0.155091,...,-0.198239,-0.101808,0.000060,0.005338,-0.025474,0.006043,-0.294483,-0.00000,-0.316869,0.000004
2,-0.026820,-0.000105,-5.511876e-05,-0.372577,0.433042,-0.017987,0.000007,-0.0,-0.0,-0.226231,...,-0.374902,-0.360699,0.000021,0.143656,0.071852,0.456135,-0.335580,-0.00000,-0.088299,0.000022
3,-0.256734,-0.000021,1.110595e-04,-0.274478,-0.005862,-0.074390,0.000039,-0.0,-0.0,-0.167387,...,0.181606,-0.000000,-0.000044,-0.081889,0.012941,0.154841,-0.316411,-0.00000,0.241368,0.000049
4,-0.198608,-0.000051,8.463812e-05,-0.292374,-0.111522,-0.049364,0.000033,-0.0,-0.0,-0.188081,...,-0.283909,-0.000000,-0.000017,-0.104336,-0.017979,0.183619,-0.238937,-0.00000,0.479296,0.000038


In [69]:
embeddings_text

array([[-0.00556899, -0.02612577,  0.02335282, ..., -0.03182795,
        -0.08463164,  0.01514528],
       [-0.0152848 ,  0.05851986, -0.00147807, ..., -0.01134741,
        -0.02245475,  0.0034332 ],
       [ 0.04777406,  0.03209759,  0.01783186, ...,  0.01024472,
         0.02251917, -0.0126278 ],
       ...,
       [ 0.01996041, -0.03764783, -0.00012554, ...,  0.00542507,
        -0.02268171, -0.05146468],
       [ 0.00063748, -0.04875954,  0.0018374 , ..., -0.01690119,
         0.00391459, -0.01053747],
       [-0.00804343, -0.01358711, -0.01385459, ...,  0.01577197,
        -0.01037218, -0.04250613]], shape=(84916, 768), dtype=float32)

In [73]:
embeddings_image

array([[-3.4998038e-01, -1.5903550e-05,  1.0470913e-04, ...,
        -4.5689940e-02,  3.4737870e-01,  6.2928215e-05],
       [ 8.5416311e-01, -1.4002803e-04,  4.4179140e-07, ...,
        -0.0000000e+00, -3.1686887e-01,  4.3420973e-06],
       [-2.6820103e-02, -1.0490522e-04, -5.5118759e-05, ...,
        -0.0000000e+00, -8.8299155e-02,  2.2088730e-05],
       ...,
       [ 8.2043946e-02, -1.5884159e-04,  1.2249734e-05, ...,
        -0.0000000e+00, -2.5362158e-01, -5.9754507e-06],
       [-3.6998615e-01, -1.6067894e-04,  6.3656230e-06, ...,
        -0.0000000e+00, -1.5052584e-01,  1.6972257e-05],
       [-1.5200017e-01, -1.5667836e-04,  3.3590961e-06, ...,
        -0.0000000e+00, -1.8097806e-01,  5.7077636e-06]],
      shape=(84916, 2560), dtype=float32)

In [ ]:
# Normalize embeddings
from sklearn.preprocessing import normalize

embeddings_text_N = normalize(embeddings_text, norm='l2')
embeddings_image_N = normalize(embeddings_image, norm='l2')

print("shape of normalized text embeddings:", embeddings_text_N.shape)
print("shape of normalized image embeddings:", embeddings_image_N.shape)

shape of normalized text embeddings: (84916, 768)
shape of normalized image embeddings: (84916, 2560)


In [72]:
embeddings_text_N

array([[-0.00556899, -0.02612577,  0.02335282, ..., -0.03182795,
        -0.08463164,  0.01514528],
       [-0.0152848 ,  0.05851986, -0.00147807, ..., -0.01134741,
        -0.02245475,  0.0034332 ],
       [ 0.04777407,  0.03209759,  0.01783186, ...,  0.01024472,
         0.02251918, -0.0126278 ],
       ...,
       [ 0.01996041, -0.03764784, -0.00012554, ...,  0.00542507,
        -0.02268171, -0.05146468],
       [ 0.00063748, -0.04875954,  0.0018374 , ..., -0.01690119,
         0.00391459, -0.01053747],
       [-0.00804343, -0.01358711, -0.01385459, ...,  0.01577197,
        -0.01037218, -0.04250613]], shape=(84916, 768), dtype=float32)

In [74]:
embeddings_image_N

array([[-2.3405232e-02, -1.0635632e-06,  7.0025108e-06, ...,
        -3.0555532e-03,  2.3231242e-02,  4.2083775e-06],
       [ 4.5399215e-02, -7.4425625e-06,  2.3481443e-08, ...,
        -0.0000000e+00, -1.6841745e-02,  2.3078474e-07],
       [-2.0735324e-03, -8.1104981e-06, -4.2613756e-06, ...,
        -0.0000000e+00, -6.8266392e-03,  1.7077376e-06],
       ...,
       [ 5.1831347e-03, -1.0034834e-05,  7.7387818e-07, ...,
        -0.0000000e+00, -1.6022569e-02, -3.7749970e-07],
       [-2.0475956e-02, -8.8923734e-06,  3.5228945e-07, ...,
        -0.0000000e+00, -8.3304755e-03,  9.3928702e-07],
       [-8.4697567e-03, -8.7304352e-06,  1.8717563e-07, ...,
        -0.0000000e+00, -1.0084463e-02,  3.1804814e-07]],
      shape=(84916, 2560), dtype=float32)

In [81]:
# fusion text and image embeddings
embeddings_combined = np.hstack((embeddings_text_N, embeddings_image_N))
print("shape of combined embeddings:", embeddings_combined.shape)

shape of combined embeddings: (84916, 3328)


In [82]:
# scale and save combined embeddings
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
embeddings_combined_scaled = scaler.fit_transform(embeddings_combined)
np.save('../data/processed/embeddings_combined_scaled.npy', embeddings_combined_scaled)
print("shape of scaled combined embeddings:", embeddings_combined_scaled.shape)

shape of scaled combined embeddings: (84916, 3328)


In [3]:
# load scaled combined embeddings
embeddings_combined_scaled = np.load('../data/processed/embeddings_combined_scaled.npy')
embeddings_combined_scaled

array([[-0.31692   ,  0.25736517,  0.97609854, ..., -0.30896267,
         0.9902836 ,  2.492436  ],
       [-0.64592874,  2.009773  , -0.3010369 , ...,  0.28457826,
        -0.87914467, -0.66928977],
       [ 1.4894472 ,  1.462756  ,  0.6921371 , ...,  0.28457826,
        -0.41193408,  0.5047167 ],
       ...,
       [ 0.5475876 ,  0.01882517, -0.23147224, ...,  0.28457826,
        -0.8409296 , -1.1528054 ],
       [-0.10674913, -0.21121903, -0.13051124, ...,  0.28457826,
        -0.48208892, -0.10611254],
       [-0.40071258,  0.5169513 , -0.93760294, ...,  0.28457826,
        -0.56391346, -0.5999255 ]], shape=(84916, 3328), dtype=float32)

In [5]:
df_model = pd.read_csv('../data/processed/df_model.csv', sep=';')
df_model.head()

,productid,imageid,prodtype,path_image,text
0,183912,528916,Jeux vidéo import,../data/raw/images/image_train/image_528916_pr...,matchbox caterpillar construction zone gbc
1,184080,1086436,Jeux vidéo import,../data/raw/images/image_train/image_1086436_p...,maya l'abeille (compatible game boy)
2,184251,234234,Jeux vidéo import,../data/raw/images/image_train/image_234234_pr...,toonstein : le château hante
3,184334,669302,Jeux vidéo import,../data/raw/images/image_train/image_669302_pr...,army men toys in space
4,184381,669220,Jeux vidéo import,../data/raw/images/image_train/image_669220_pr...,international track & field


In [6]:
X = embeddings_combined_scaled
y = df_model['prodtype'].to_numpy()

print("shape of X:", X.shape)
print("shape of y:", y.shape)


shape of X: (84916, 3328)
shape of y: (84916,)


### Labels encoding

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = np.asarray(le.fit_transform(y))
num_classes = len(le.classes_)
print("encoded labels:", y_encoded[:5])
print("Number of classes:", num_classes)
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))


encoded labels: [12 12 12 12 12]
Number of classes: 27
Label mapping: {'Accessoires jeux vidéo': 0, 'Accessoires pour la maison': 1, 'Alimentaire': 2, 'Cartes à jouer': 3, 'Décoration intérieure': 4, 'Figurines wargames': 5, 'Goodies geek': 6, 'Jardinage & bricolage': 7, 'Jeux de société': 8, 'Jeux et consoles rétro': 9, 'Jeux geek': 10, 'Jeux vidéo dématérialisés': 11, 'Jeux vidéo import': 12, 'Jouets enfance': 13, 'Jouets enfants': 14, 'Jouets petite enfance': 15, 'Livres adultes': 16, 'Livres illustrés': 17, 'Livres jeunesse': 18, 'Magazines': 19, 'Matériel enfance': 20, 'Mobilier & literie intérieure': 21, 'Mobilier de jardin & cuisine': 22, 'Modélisme & jouets télécommandés': 23, 'Papeterie': 24, 'Piscines & spas': 25, 'Produits pour animaux': 26}


### Data spliting with stratify

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (59441, 3328) Val: (12737, 3328) Test: (12738, 3328)


### **Class Imbalance Management**

A significant class imbalance was observed in the dataset. For example:

* **Majority class:** *“Piscines & Spas”* (over 10,000 products)
* **Minority class:** *“Figurines Wargames”* (fewer than 1,000 products)

To mitigate this issue, we apply **class weighting**, assigning higher weights to under-represented classes and lower weights to over-represented ones.

This approach increases the penalty for misclassifying rare classes, preventing the model from disproportionately favoring majority classes and ensuring more balanced learning across all categories.


In [ ]:
# imbalance handling with class weights
from sklearn.utils import compute_class_weight

classes = np.unique(y_train)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = {c: w for c, w in zip(classes, class_weights_arr)}

weights_train = np.array([class_weights[c] for c in y_train])
weights_val = np.array([class_weights[c] for c in y_val])
weights_test = np.array([class_weights[c] for c in y_test])

print(classes)
print(class_weights)
print(weights_train[:10])
print(weights_val[:10])
print(weights_test[:10])

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26]
{np.int64(0): np.float64(1.8704490386733377), np.int64(1): np.float64(0.730915842801633), np.int64(2): np.float64(3.917292737577435), np.int64(3): np.float64(0.7956337255216909), np.int64(4): np.float64(0.629905155513167), np.int64(5): np.float64(4.114987885081343), np.int64(6): np.float64(1.1772826302238066), np.int64(7): np.float64(1.260170874939049), np.int64(8): np.float64(1.5193364517036014), np.int64(9): np.float64(3.7826778668703067), np.int64(10): np.float64(2.212581425646752), np.int64(11): np.float64(3.6090467516697027), np.int64(12): np.float64(1.2537121403864), np.int64(13): np.float64(0.6457959866584096), np.int64(14): np.float64(1.262338600067958), np.int64(15): np.float64(0.9702593735207221), np.int64(16): np.float64(1.0094078489310034), np.int64(17): np.float64(1.138912839378437), np.int64(18): np.float64(0.6587428242126028), np.int64(19): np.float64(0.6607198434929528), np.int64(20): np

### LGBMClassifier

In [18]:
import lightgbm as lgb
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=num_classes,
    n_estimators=8000,
    learning_rate=0.00762729714650079, #0.01,
    max_depth=13, #-1,
    num_leaves=84, #64,
    subsample=0.8461855547052801, #0.9,
    colsample_bytree=0.926289123459471, #0.9,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    min_child_samples=81,
    reg_alpha=0.03569832875035132,
    reg_lambda=3.6345239734358,
    min_split_gain=0.3453405161243368,
    verbosity=-1,
    importance_type='gain'
)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=50)
]

lgbm.fit(
    X_train,
    y_train,
    sample_weight=weights_train,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[weights_val],
    eval_metric=["multi_logloss", "multi_error"],
    callbacks=callbacks
)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 2.09759	valid_0's multi_error: 0.322924
[100]	valid_0's multi_logloss: 1.65881	valid_0's multi_error: 0.304747
[150]	valid_0's multi_logloss: 1.40672	valid_0's multi_error: 0.289651
[200]	valid_0's multi_logloss: 1.2456	valid_0's multi_error: 0.278238
[250]	valid_0's multi_logloss: 1.13635	valid_0's multi_error: 0.268816
[300]	valid_0's multi_logloss: 1.05951	valid_0's multi_error: 0.261012
[350]	valid_0's multi_logloss: 1.00431	valid_0's multi_error: 0.258415
[400]	valid_0's multi_logloss: 0.964094	valid_0's multi_error: 0.25537
[450]	valid_0's multi_logloss: 0.933075	valid_0's multi_error: 0.250997
[500]	valid_0's multi_logloss: 0.909426	valid_0's multi_error: 0.250888
[550]	valid_0's multi_logloss: 0.890992	valid_0's multi_error: 0.248837
[600]	valid_0's multi_logloss: 0.875948	valid_0's multi_error: 0.246871
[650]	valid_0's multi_logloss: 0.863317	valid_0's multi_error: 0.246464
[700]	valid_0

,boosting_type,'gbdt'
,num_leaves,84
,max_depth,13
,learning_rate,0.00762729714650079
,n_estimators,8000
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.3453405161243368
,min_child_weight,0.001
,min_child_samples,81


In [23]:
# save lgbm model
import joblib
joblib.dump(lgbm, '../models/best_lgbm.pkl')

['../models/best_lgbm.pkl']

In [33]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = lgbm.predict(X_val)  

print(classification_report(y_val, y_pred, target_names=le.classes_, digits=3))
def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

#plot_confusion_matrix(y_val, y_pred, labels=le.classes_)

/data/MultiModalAI-MLOps/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


                                  precision    recall  f1-score   support

          Accessoires jeux vidéo      0.799     0.790     0.794       252
      Accessoires pour la maison      0.877     0.895     0.886       645
                     Alimentaire      0.810     0.843     0.826       121
                  Cartes à jouer      0.933     0.943     0.938       593
           Décoration intérieure      0.743     0.740     0.741       749
              Figurines wargames      0.763     0.504     0.607       115
                    Goodies geek      0.701     0.757     0.728       400
           Jardinage & bricolage      0.731     0.659     0.693       375
                 Jeux de société      0.594     0.405     0.482       311
          Jeux et consoles rétro      0.890     0.904     0.897       125
                       Jeux geek      0.838     0.803     0.820       213
       Jeux vidéo dématérialisés      0.880     0.893     0.886       131
               Jeux vidéo import     

The model achieves **78.9% accuracy across 27 product categories**, with strong performance on well-defined segments such as *Piscines & Spas*, *Modélisme*, *Papeterie*, and *Jeux vidéo*.  

However, several categories remain more difficult to classify due to **semantic overlap** or **limited representation**, including *Jeux de société*, *Figurines wargames*, and *Jouets enfance*.  

The performance gaps are primarily explained by (1) class imbalance, (2) ambiguous or noisy product descriptions, and (3) limitations in the current text & image embeddings.  

### **Current limitations: CPU-bound training**

The training pipeline currently relies mostly on **CPU**, which imposes several constraints:

* limits the **size and power of language models** used for text embedding
* restricts the **complexity** of the classifier
* prevents training or fine-tuning **large neural models**
* slows down experimentation and hyperparameter tuning

### **Key improvement opportunities**

* **Integration of richer structured features**, such as:
  *brand*, *price range*, *parent category*, *keyword indicators*

* **Using more powerful modern embeddings**, such as multilingual DistilUSE, or domain-adapted embeddings for e-commerce descriptions

* **Using stronger GPU-based text & image embeddings**

* **Enhanced class rebalancing strategies**, including synthetic text/image augmentation, or hierarchical classification to reduce confusion between close categories

With these enhancements, a **3–7% improvement in overall accuracy** is realistically attainable, especially for minority or semantically similar classes.

## Optuna : search best params for LGBMClassifier

In [ ]:
import numpy as np
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier
from optuna.integration import LightGBMPruningCallback  


num_classes = len(np.unique(y_train))

def objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": num_classes,
        "n_estimators": 8000,      
        "n_jobs": -1,              # CPU cores
        "force_col_wise": True,
        "random_state": 42,
        "verbosity": -1,           

        # Hyperparams à optimiser
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", 4, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 120),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }

    model = LGBMClassifier(**params)

    # Callback de pruning Optuna sur la métrique de validation
    pruning_callback = LightGBMPruningCallback(
        trial,
        metric="multi_error"   
    )

    # Entraînement avec early stopping + pruning
    model.fit(
        X_train,
        y_train,
        sample_weight=weights_train,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[weights_val],  
        eval_metric=["multi_logloss", "multi_error"],
        callbacks=[
            pruning_callback,
            lgb.early_stopping(stopping_rounds=100, verbose=False),
        ],
    )

    # Minimise l'erreur multi-classes sur la validation
    score = model.best_score_["valid_0"]["multi_error"]
    return score

n_trials = 50  # nombre d'essais d'optimisation

study = optuna.create_study(direction="minimize")
study.optimize(
    objective,
    n_trials=n_trials,
    show_progress_bar=True, 
)

print("Meilleur essai :")
print(f"  Valeur (min multi_error): {study.best_value:.5f}")
print("  Meilleurs hyperparamètres :")
print(study.best_params)


[I 2025-11-18 00:24:20,149] A new study created in memory with name: no-name-13a16fef-9846-4756-96ec-15c27ffe1664
Best trial: 0. Best value: 0.236583:   2%|▏         | 1/50 [44:59<36:44:13, 2699.04s/it]

[I 2025-11-18 01:09:19,191] Trial 0 finished with value: 0.23658305159802323 and parameters: {'learning_rate': 0.008849985490599987, 'num_leaves': 65, 'max_depth': 7, 'min_child_samples': 116, 'subsample': 0.9100336836281171, 'colsample_bytree': 0.8700241048096364, 'reg_alpha': 0.00308682209416246, 'reg_lambda': 0.05488779082962207, 'min_split_gain': 0.17730085378727034}. Best is trial 0 with value: 0.23658305159802323.


Best trial: 1. Best value: 0.23433:   4%|▍         | 2/50 [1:24:34<33:27:05, 2508.87s/it] 

[I 2025-11-18 01:48:54,939] Trial 1 finished with value: 0.2343296929512643 and parameters: {'learning_rate': 0.00762729714650079, 'num_leaves': 84, 'max_depth': 13, 'min_child_samples': 81, 'subsample': 0.8461855547052801, 'colsample_bytree': 0.926289123459471, 'reg_alpha': 0.03569832875035132, 'reg_lambda': 3.6345239734358, 'min_split_gain': 0.3453405161243368}. Best is trial 1 with value: 0.2343296929512643.


Best trial: 1. Best value: 0.23433:   6%|▌         | 3/50 [1:34:35<21:22:54, 1637.76s/it]

...

[I 2025-11-18 04:54:08,031] Trial 47 pruned. Trial was pruned at iteration 1.


Best trial: 1. Best value: 0.23433:  98%|█████████▊| 49/50 [4:29:57<01:54, 114.81s/it]

[I 2025-11-18 04:54:17,341] Trial 48 pruned. Trial was pruned at iteration 1.


Best trial: 1. Best value: 0.23433: 100%|██████████| 50/50 [4:31:06<00:00, 325.32s/it]

[I 2025-11-18 04:55:26,282] Trial 49 pruned. Trial was pruned at iteration 22.
Meilleur essai :
  Valeur (min multi_error): 0.23433
  Meilleurs hyperparamètres :
{'learning_rate': 0.00762729714650079, 'num_leaves': 84, 'max_depth': 13, 'min_child_samples': 81, 'subsample': 0.8461855547052801, 'colsample_bytree': 0.926289123459471, 'reg_alpha': 0.03569832875035132, 'reg_lambda': 3.6345239734358, 'min_split_gain': 0.3453405161243368}
